# Training a CellPose network

Run one deterministic CPU optimizer step without downloading data or weights.
This is a minimal starting point for a plain PyTorch training loop, not a training
recipe or a claim about accuracy. There are two synthetic objects and one foreground
class. Background is type 0; both instances have type 1. Instance IDs are preserved
for flow-target generation. Images are float32 BCHW in [0, 1]; class targets are
int64 BHW and flow targets are float32 B2HW.

For real data, `TrainDatasetFolder` reads paired image and `.mat` files with `inst`
and `type` arrays. `TrainDatasetH5` is the HDF5 alternative and additionally needs
PyTables. Both training datasets require Albumentations and aligned image/mask
transforms plus `ApplyEach([CellposeTransform()], as_list=True)` for flow targets.
Folder pairs use sorted file order: verify identifiers, shapes and class mapping.
After geometric augmentation, verify that the flow directions still match the
transformed objects. Split patients/slides before making patches; never use these
duplicate smoke samples as a real validation split.


In [ ]:
import numpy as np
import torch
from cellseg_models_pytorch.models.cellpose.cellpose_unet import cellpose_nuclei
from cellseg_models_pytorch.transforms.functional.cellpose import gen_flow_maps

torch.manual_seed(0)
torch.set_num_threads(2)
device = torch.device("cpu")
rows, cols = np.mgrid[:64, :64]
instances = np.zeros((64, 64), dtype=np.int32)
instances[(rows - 24)**2 + (cols - 24)**2 < 8**2] = 1
instances[(rows - 44)**2 + (cols - 44)**2 < 6**2] = 2
# Synthetic RGB pixels are only a smoke fixture, not independent real training data.
image = np.repeat((instances > 0)[..., None], 3, axis=2).astype(np.float32)
images = torch.from_numpy(image.transpose(2, 0, 1)).unsqueeze(0).repeat(2, 1, 1, 1)
types = torch.from_numpy((instances > 0).astype(np.int64)).unsqueeze(0).repeat(2, 1, 1)
flows = torch.from_numpy(gen_flow_maps(instances)).unsqueeze(0).repeat(2, 1, 1, 1)
assert flows.shape == (2, 2, 64, 64) and torch.isfinite(flows).all()
model = cellpose_nuclei(2, enc_name="resnet18", enc_pretrain=False).to(device)


The raw `cellpose_nuclei` network supports autograd. Its `nuc` output carries
`type_map` class logits and `aux_map` vertical/horizontal flows. This example combines
ordinary cross entropy and flow MSE equally. The library also provides configurable
loss components; select weighting and metrics for your dataset rather than treating
this one-step example as a scientifically calibrated recipe.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
parameter = next(model.parameters())
before = parameter.detach().clone()
model.train()
optimizer.zero_grad(set_to_none=True)
prediction = model(images.to(device))["nuc"]
loss = torch.nn.functional.cross_entropy(prediction.type_map, types.to(device))
loss = loss + torch.nn.functional.mse_loss(prediction.aux_map, flows.to(device))
assert torch.isfinite(loss)
loss.backward()
gradients = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
assert gradients and all(torch.isfinite(gradient).all() for gradient in gradients)
optimizer.step()
print("Smoke training loss:", float(loss.detach()))
assert not torch.equal(before, parameter.detach())


In [ ]:
model.eval()
with torch.inference_mode():
    output = model(images.to(device))["nuc"]
    validation_loss = torch.nn.functional.cross_entropy(output.type_map, types.to(device))
    assert torch.isfinite(validation_loss)
print("Smoke evaluation loss (same synthetic input):", float(validation_loss))


For a real run, iterate separate train/validation loaders over multiple epochs,
use instance reconstruction and instance metrics for validation, and save resumable
state as in the fine-tuning example. Choose device/precision explicitly. GPU/AMP,
data augmentation and full-dataset training are not validated by this smoke run.
